# Sorting

*Run a cell with ▶ (or Shift+Enter): Sphinx compiles and runs it on your computer. Change the code and run it again: this notebook is your own copy.*

**After this unit you can:**
- Trace merge sort's merges and explain why it takes O(n log n)
- Write the merge step, and reuse it to count inversions
- Sort by a key with a comparator, then sweep the sorted data in one pass
- Use counting sort when the values fall in a small range

**Exercises** in this notebook end with `check(...)` lines: minimal tests that print ✓ when your code is right. Run the next cell once first, so they can use `check`.

In [ ]:
// shared: check(...) prints ✓ or ✗; the exercises below use it
void check(String what, Object actual, Object expected) {
    if (Objects.equals(actual, expected)) {
        IO.println("✓ " + what);
    } else {
        IO.println("✗ " + what + ": expected " + expected + ", got " + actual);
    }
}

## In short

Sorting makes data easy to search, group and compare, and the main O(n log n) sorts use one of two ideas.

**Merge sort** merges sorted runs. Merging two sorted lists takes one comparison per value taken, so a pass over n values costs about n. Start with runs of 1 and double their size every pass: after log₂ n passes there is one sorted run, for O(n log n) in total, always.

#### 🎞️ Merging two sorted runs

![Merging 2 5 8 with 1 6 9: the smaller front value goes out each time, giving 1 2 5 6 8 9](merge-steps.svg)

![Merge sort's passes on 8 values](passes.svg)

**O(n²) against O(n log n):** selection sort and `Arrays.sort` on the same random numbers. Double `n` and compare.

In [ ]:
int n = 20_000;
int[] a = new Random(2).ints(n, 0, 1_000_000).toArray();
int[] b = a.clone();

long start = System.nanoTime();
for (int i = 0; i < n; i++) { // selection sort
    int min = i;
    for (int j = i + 1; j < n; j++) {
        if (a[j] < a[min]) {
            min = j;
        }
    }
    int t = a[i];
    a[i] = a[min];
    a[min] = t;
}
long middle = System.nanoTime();
Arrays.sort(b);
long end = System.nanoTime();

IO.println("selection sort: " + (middle - start) / 1_000_000 + " ms");
IO.println("Arrays.sort:    " + (end - middle) / 1_000_000 + " ms");
IO.println("same result: " + Arrays.equals(a, b));

### ✏️ Exercise: Merge two sorted arrays

The step at the heart of merge sort: build one sorted array from two sorted ones, always taking the smaller front value.

In [ ]:
int[] merge(int[] a, int[] b) {
    // TODO: take the smaller front value until one side runs out, then copy the rest
    return new int[0];
}

void main() {
    check("mixed", Arrays.toString(merge(new int[] {2, 5, 8}, new int[] {1, 6, 9})), "[1, 2, 5, 6, 8, 9]");
    check("one empty", Arrays.toString(merge(new int[] {}, new int[] {3, 4})), "[3, 4]");
    check("repeats", Arrays.toString(merge(new int[] {1, 1}, new int[] {1})), "[1, 1, 1]");
}

<details>
<summary>Show an answer</summary>

```java
int[] merge(int[] a, int[] b) {
    int[] out = new int[a.length + b.length];
    int i = 0, j = 0, k = 0;
    while (i < a.length && j < b.length) {
        out[k++] = a[i] <= b[j] ? a[i++] : b[j++];
    }
    while (i < a.length) {
        out[k++] = a[i++];
    }
    while (j < b.length) {
        out[k++] = b[j++];
    }
    return out;
}

void main() {
    check("mixed", Arrays.toString(merge(new int[] {2, 5, 8}, new int[] {1, 6, 9})), "[1, 2, 5, 6, 8, 9]");
    check("one empty", Arrays.toString(merge(new int[] {}, new int[] {3, 4})), "[3, 4]");
    check("repeats", Arrays.toString(merge(new int[] {1, 1}, new int[] {1})), "[1, 1, 1]");
}
```

</details>

**Quicksort** picks a pivot, moves smaller values to its left and bigger ones to its right (a **partition**), then sorts each side. It is O(n log n) on average, but a bad pivot every time, such as the first element of a sorted array, makes it O(n²).

**A bad pivot every time.** This quicksort always picks the first element. Count its comparisons on random and on already sorted data.

In [ ]:
// shared: quicksort with the first element as the pivot, counting comparisons
long comparisons = 0;

void quicksort(int[] a, int lo, int hi) {
    if (lo >= hi) {
        return;
    }
    int pivot = a[lo];
    int i = lo;
    for (int j = lo + 1; j <= hi; j++) {
        comparisons++;
        if (a[j] < pivot) {
            i++;
            int t = a[i];
            a[i] = a[j];
            a[j] = t;
        }
    }
    int t = a[lo];
    a[lo] = a[i];
    a[i] = t;
    quicksort(a, lo, i - 1);
    quicksort(a, i + 1, hi);
}

In [ ]:
int n = 3000;
int[] random = new Random(3).ints(n, 0, 1_000_000).toArray();
quicksort(random, 0, n - 1);
IO.println(String.format("random: %,d comparisons", comparisons));

comparisons = 0;
int[] sorted = new int[n];
for (int i = 0; i < n; i++) {
    sorted[i] = i;
}
quicksort(sorted, 0, n - 1);
IO.println(String.format("sorted: %,d comparisons (n²/2 = %,d)", comparisons, (long) n * n / 2));

When the values are small whole numbers, such as scores from 0 to 100, **counting sort** doesn't compare at all: it counts how often each value appears, in O(n + k) for k possible values.

### ✏️ Exercise: Insertion sort

Sort the array in place: take each value in turn and slide it left past the bigger values before it. It is O(n²), but very fast on almost sorted data.

In [ ]:
/** Sorts a copy with insertionSort and returns it as text. */
String sortedCopy(int[] a) {
    int[] copy = a.clone();
    insertionSort(copy);
    return Arrays.toString(copy);
}

void insertionSort(int[] a) {
    // TODO: for each i, move a[i] left while the value before it is bigger
}

void main() {
    check("sorts", sortedCopy(new int[] {5, 2, 9, 1}), "[1, 2, 5, 9]");
    check("repeats", sortedCopy(new int[] {3, 1, 3, 2}), "[1, 2, 3, 3]");
    check("already sorted", sortedCopy(new int[] {1, 2, 3}), "[1, 2, 3]");
}

<details>
<summary>Show an answer</summary>

```java
/** Sorts a copy with insertionSort and returns it as text. */
String sortedCopy(int[] a) {
    int[] copy = a.clone();
    insertionSort(copy);
    return Arrays.toString(copy);
}

void insertionSort(int[] a) {
    for (int i = 1; i < a.length; i++) {
        int value = a[i];
        int j = i - 1;
        while (j >= 0 && a[j] > value) {
            a[j + 1] = a[j];
            j--;
        }
        a[j + 1] = value;
    }
}

void main() {
    check("sorts", sortedCopy(new int[] {5, 2, 9, 1}), "[1, 2, 5, 9]");
    check("repeats", sortedCopy(new int[] {3, 1, 3, 2}), "[1, 2, 3, 3]");
    check("already sorted", sortedCopy(new int[] {1, 2, 3}), "[1, 2, 3]");
}
```

</details>

In Java, `Arrays.sort` on an `int[]` uses a quicksort, and on objects a merge sort driven by a **comparator**, such as `Comparator.comparingInt(Meeting::start)`. That sort is **stable**: equal elements keep their order.

Watch out: a comparator written as `a - b` can overflow. Use `Integer.compare(a, b)`.

**Comparators:** by length, then alphabetically, and reversed. The last line shows the `a - b` overflow.

In [ ]:
List<String> words = new ArrayList<>(List.of("pear", "fig", "apple", "kiwi", "date"));
words.sort(Comparator.comparingInt(String::length).thenComparing(Comparator.naturalOrder()));
IO.println(words);
words.sort(Comparator.comparingInt(String::length).reversed());
IO.println(words);

List<Integer> numbers = new ArrayList<>(List.of(2_000_000_000, -1_000_000_000, 0));
numbers.sort((a, b) -> a - b); // overflows
IO.println("with a - b:            " + numbers);
numbers.sort(Integer::compare);
IO.println("with Integer.compare:  " + numbers);

## Readings

- [Sorting](https://visualgo.net/en/sorting) (VisuAlgo (National University of Singapore), 20 min): Pick Merge Sort, Quick Sort and Counting Sort from the top menu and run each on your own short array. Watch the merge take the smaller front value, the partition move values around the pivot, and counting sort never compare two values.
- [Writing and Combining Comparators](https://dev.java/learn/language/fp/lambdas/writing-comparators/) (dev.java (Oracle), 8 min): Comparator.comparing and comparingInt build a comparator from a key, thenComparing breaks ties with a second key, and reversed() flips the order. Note why (i1 - i2) is a bad comparator.

## Check yourself

1. How many passes does bottom-up merge sort make on 1,000 values?
2. Why can't counting sort sort names?
3. What does a stable sort keep, and when does that matter?

## Practice

The notebook covered the basics. Now apply them in the unit's challenges, in the Sphinx sidebar, where Submit runs every test, hidden ones included:

- **Exam Scores** · Easy
- **Merge Trace** · Medium
- **Calendar Merge** · Medium
- **Ranking Distance** · Hard